In [ ]:
import os
from dotenv import load_dotenv
from datetime import datetime
from zoneinfo import ZoneInfo
from typing import Dict,Any
from simpleeval import SimpleEval
import requests
import logging 

load_dotenv()  #用于从.env文件加载环境到当前运行环境

#参数timezone,.返回值返回一个字典,键位str,值为任意类型
""" 获取指定时区的当前时间
Args:timezone:时区名称,类型为str,默认值为Asia/Xian 
Returns:统一格式返回,包含status/data/message。成功时data为时间字符串,失败时 data为none
Raises:无,所有异常捕获后以fail状态返回。  
Example:get_current_time("America/california")"""
def get_current_time(timezone:str="Asia/Shanghai") ->Dict[str,Any]:#及返回{"time":"2026","data":"2026-7-01"}
    now=datetime.now(ZoneInfo(timezone))  #ZoneInfo(timezone)，用传入的时区变量
    try:
        return {"status": "success","data": now.strftime("%Y-%m-%d %H:%M:%S"),"message": "success"}
    except Exception as e:
        return {"status":"fail","data":None,"message":f"时区错误:{str(e)}"}
print(get_current_time())
"""计算数学表示表达式
Args:expression:数字表达式字符串,如"1+2+3" 
Returns:统一格式返回,包含status/data/message 成功时返回data,失败时data为none
Raises:无,所有异常捕获后以fail状态返回。  
Example:calculate("1+2*3")  """
calc=SimpleEval()
def calculate(expression: str) -> Dict[str, Any]:
    try:
        result=calc.eval(expression)
        return {"status":"success","data":result,"message":"计算成功"}
    except Exception as e:
        return {"status":"fail","data":None,"message":f"计算错误:{str(e)}"}
print(calculate("2+3"))
print(calculate("2/0"))

#获取环境变量
api_key=os.getenv("weather_Api_Key")
logger=logging.getLogger(__name__)
"""获取指定城市实时天气 
Args:city:城市名称,如"北京"  
Returns:统一格式返回,包含status/data/message  成功时data包含city/temp/feels_like/humidity/weather/wind_speed,失败时data为None
Raises:无,所有异常捕获后以fail状态返回。  
Example:get_weather("xi'an")  """
def get_weather(city:str)->Dict[str,Any]:
    if not api_key:
        logger.warning("Weather_Api_Key未配置")
        return {"status":"fail","data":None,"message":"未配置OpenWeatherMap API Key"}
    url="https://api.openweathermap.org/data/2.5/weather"
    params={ 
        "q":city,"appid":api_key,"units":"metric","lang":"zh_cn" }#metric表示摄氏度
    try:
        resp=requests.get(url,params=params,timeout=100)
        data=resp.json()
        if resp.status_code==200:
            return { "status":"success","data":{"city":data.get("name"),"temp":data["main"]["temp"],"feels_like":data["main"]["feels_like"],
            "humidity":data["main"]["humidity"],"weather":data["weather"][0]["description"],"wind_speed":data["wind"]["speed"]
            },"message":"success"  
            }
        else: 
            logger.error(f"Weather ApI error:{data.get('message','unknown')}")
            return {"status":"fail","data":None,"message":data.get("message","查询失败")} 
    except requests.Timeout:
        logger.warning(f"Weather API timeout for city:{city}")
        return {"status":"fail",data:None,"message":f"{city}天气服务暂不可用"}
    except Exception as e:
        logger.error(f"weather API error:{str(e)}")
        return {"status":"fail","data":None,"message":str(e)}
result=get_weather("Xi'an")
print(result)

{'status': 'success', 'data': '2026-07-01 19:07:06', 'message': 'success'}
{'status': 'success', 'data': 5, 'message': '计算成功'}
{'status': 'fail', 'data': None, 'message': '计算错误:division by zero'}
{'status': 'success', 'data': {'city': "Xi'an", 'temp': 33.05, 'feels_like': 32.69, 'humidity': 34, 'weather': '阴，多云', 'wind_speed': 4.13}, 'message': 'success'}


In [ ]:
#模拟FMM和RMM
import hanlp
from transformers import BertTokenizer

# 1. HanLP 兼容补丁（适配新版 transformers）
def patch_hanlp_tokenizer():
    """为 HanLP 打补丁，使其兼容新版 transformers"""
    if not hasattr(BertTokenizer, "encode_plus"):
        def encode_plus(self, text, **kwargs):
            return self(text, **kwargs)
        BertTokenizer.encode_plus = encode_plus
        print("[补丁] 已为 BertTokenizer 添加 encode_plus 方法")
    else:
        print("[补丁] 无需补丁，当前版本兼容")

patch_hanlp_tokenizer()

#加载Hanlp分词模型
tokenizer=hanlp.load(hanlp.pretrained.tok.FINE_ELECTRA_SMALL_ZH)
print("[模型]HanLP分词模型加载完成")

#手写词典分词方法
"""正向最大匹配
    Args:句子,字典,分词最大长度   returns:返回最终分词结果
    双层循环处理   
       第一层循环:i正向走
       第二层循环:j的初值是从i出发的字符串长度,每次循环递减1,最低应为1或者称为0的由开集。
"""
def fmm(sentence,dictionary,max_len=5):
    result=[];i=0
    n=len(sentence)
    while i <n:
        matched=False
        for j in range(min(max_len,n-i),0,-1):#没到句尾,长度为max_len,则j in 范围[5,0)步长-1 及5,4,3,2,1
            word=sentence[i:i+j]
            if word in dictionary:
                result.append(word)
                i+=j  #直接让i
                matched=True
                break
        if matched==False:
            result.append(sentence[i])
            i+=1
    return result
"""
    逆向最长匹配
    Args:句子,字典,分词最大长度   returns:返回最终分词结果
    双层循环:
    第一层:i从n开始,即数组末尾的下一个下标开始,递减
    第二层:仍然是长度
"""
def rmm(sentence,dictionary,max_len=5):
    result=[]
    i=len(sentence)
    while(i>0):
        matched=False
        for length in range(min(max_len,i),0,-1):
            word=sentence[i-length:i]
            if word in dictionary:
                result.append(word)
                i-=length
                matched=True
                break
        if not matched:
            result.append(sentence[i-1])
            i-=1
    result.reverse()
    return result
word_dict = {
    "南京", "南京市", "长江", "大桥",
    "研究", "生物", "生物化学", "化学", "研究生",
    "项目","研究","目的"
    }

test_sentences = [
    "南京市长江大桥",
    "研究生物化学",
    "项目的研究"
]

print("=" * 80)
print("对比结果：手写FMM vs 手写RMM vs HanLP")

for sent in test_sentences:
    fmm_res=fmm(sent,word_dict,5)
    rmm_res=rmm(sent,word_dict,5)
    hanlap_res=tokenizer(sent)

    print(f"\n句子:{sent}")
    print(f" FMM :{'/'.join(fmm_res)}")
    print(f" RMM :{'/'.join(rmm_res)}")
    print(f"HanLp:{'/'.join(hanlap_res)}")

[补丁] 无需补丁，当前版本兼容


[模型]HanLP分词模型加载完成
对比结果：手写FMM vs 手写RMM vs HanLP

句子:南京市长江大桥
 FMM :南京市/长江/大桥
 RMM :南京市/长江/大桥
HanLp:南京市/长江/大桥

句子:研究生物化学
 FMM :研究生/物/化学
 RMM :研究/生物化学
HanLp:研究/生物/化学

句子:项目的研究
 FMM :项目/的/研究
 RMM :项/目的/研究
HanLp:项目/的/研究
